# Setting Up the Environment

## Importing Libraries:

- **langchain-google-genai**: A LangChain integration that enables seamless interaction with Google's AI models, such as Gemini-Pro, for advanced language processing and AI workflows.  
- **langchain.prompts import PromptTemplate**: A utility in LangChain for creating dynamic, reusable text prompts with placeholders for structured AI interactions.  
- **requests**: A popular library for making HTTP requests in Python.  
- **json**: Used for parsing JSON data, which is common in API responses.  
- **from google.colab**: Used to securely store and retrieve user-specific data in Google Colab.  


In [ ]:
%pip install langchain
%pip install langchain langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from google.colab import userdata



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 345.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.1 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.4 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.16 which is incompatible.


# Secure API key handling

In [ ]:
LLUMOAI_KEY = userdata.get("LLUMO_API_KEY")  # Ensure this is set in the environment(Visit https://app.llumo.ai/ to get your own Api key)
GOOGLE_KEY = userdata.get("GOOGLE_API_KEY")

# Ensure API key is available
if not GOOGLE_KEY:
    raise ValueError("Missing Google AI API key. Set GOOGLE_API_KEY as an environment variable.")

# Initialize Google Gemini-Pro model
chatModel = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=GOOGLE_KEY)


# Defining Prompt

In [ ]:
# Defining the Prompt
promptTemplate = PromptTemplate.from_template("Explain {topic} in a simple and easy-to-understand way.")


# Creating Chain

In [ ]:
# Create a chain
chain = promptTemplate | chatModel

# Getting Respone

In [ ]:
# Run the chain with a topic
topic = "quantum physics"
response = chain.invoke({"topic": topic})

In [ ]:
response.content

'Okay, let\'s try to explain quantum physics in a simple way, avoiding complex math as much as possible. Imagine the world isn\'t like a smooth, predictable movie, but more like a box of LEGO bricks.\n\n**The Core Ideas:**\n\n*   **Quantization:** Think of it like LEGOs.  You can\'t have half a LEGO brick. You can only have whole bricks.  In the quantum world, energy, momentum, and angular momentum (how things spin) are also "quantized."  They can only exist in specific, discrete amounts, like those LEGO bricks.  It\'s not a continuous flow, but rather pre-defined packets.\n\n*   **Duality (Wave-Particle):**  Imagine a LEGO brick that can sometimes act like a wave, spreading out and interfering with other LEGOs.  Quantum objects, like electrons and photons (light particles), can act like both particles (like tiny bullets) *and* waves (like ripples in water).  It\'s not that they *are* both at the same time, but rather they exhibit properties of both, depending on how we observe them. I

# Evaluating Responses with Llumo

##Evaluate the Gemini Response:
-We evaluate the response generated by the Google Gemini Model, using the Llumo evaluation Api.</br>
We call the Api the example prompt and the openai_output.</br>
We check if the evaluation was successful:</br>
If successful, we print the Llumo evaluation results.</br>
If the evaluation fails, we print an error message and indicate that the original prompt can be used if evaluation fails.

In [ ]:
import requests
# Define the endpoint, headers, and payload
LLUMO_ENDPOINT = "https://app.llumo.ai/api/create-eval-analytics"
headers = {
    "Authorization": f"Bearer {LLUMOAI_KEY}", # Replace with your LLumo API key it will look like this "Bearer A1B2C3"
    "Content-Type": "application/json"
}
payload = {
    "prompt": promptTemplate.template,
    "input": {"topic":"quantum physics"},
    "output": response.content,
    "analytics": ["Confidence"] # ANALYTICS NAME are Confidence,Clarity,Context.....etc.
}
# Make the API request
response = requests.post(LLUMO_ENDPOINT, json=payload, headers=headers)
print(response)
try:
    result = response.json()  # Parse the JSON response
    print("statusCode : ", result['data']['statusCode'])
    print("message : ",result['data']['message'])
    # Extract the 'data' part
    data = result.get('data', {})
    print("Analytics:", data)
    # Return the data and a success flag

except Exception as e:
  print(e)

<Response [200]>
statusCode :  200
message :  SUCCESS
Analytics: {'data': '{"analyticsScore": {"confidence": 75, "context": 90, "clarity": 85, "overallScore": 83}, "reasoning": {"confidence": ["The output displays confidence through its clear and structured explanation of complex concepts.", "The use of analogies (LEGO bricks) and confident assertions (\'It\'s not that they *are* both at the same time, but rather they exhibit properties of both...\') contribute to a sense of assurance.", "However, the occasional use of phrases like \'It\'s a bit like saying...\' slightly undermines the overall feeling of complete certainty."], "context": ["The output directly addresses all core aspects of the prompt: explaining quantum physics in a simple and easy-to-understand way.", "It systematically covers key concepts like quantization, duality, superposition, uncertainty, and entanglement.", "The explanation goes beyond superficial information, providing meaningful insights and analogies to aid u

In [ ]:
data

{'data': '{"analyticsScore": {"confidence": 75, "context": 90, "clarity": 85, "overallScore": 83}, "reasoning": {"confidence": ["The output displays confidence through its clear and structured explanation of complex concepts.", "The use of analogies (LEGO bricks) and confident assertions (\'It\'s not that they *are* both at the same time, but rather they exhibit properties of both...\') contribute to a sense of assurance.", "However, the occasional use of phrases like \'It\'s a bit like saying...\' slightly undermines the overall feeling of complete certainty."], "context": ["The output directly addresses all core aspects of the prompt: explaining quantum physics in a simple and easy-to-understand way.", "It systematically covers key concepts like quantization, duality, superposition, uncertainty, and entanglement.", "The explanation goes beyond superficial information, providing meaningful insights and analogies to aid understanding.", "The inclusion of real-world applications further